In [ ]:
import json
import re
import pathlib

import numpy as np
import pandas as pd

def read_log_file(path: str | pathlib.Path) -> tuple[str, str]:
    path = pathlib.Path(path)
    raw = path.read_bytes()
    try:
        return raw.decode("utf-8"), "utf-8"
    except UnicodeDecodeError:
        return raw.decode("ISO-8859-1"), "ISO-8859-1"

LOG_PATTERN = re.compile(
    r"^\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\s+"
    r"\[(?P<register_id>[^\]]+)\]\s+"
    r"item=(?P<item>.+?)\s+"
    r"price=(?P<price>\d+(?:\.\d+)?)\s+"
    r"(?P<currency>[A-Z]{3})\s+"
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)\s*$"
)

def parse_log(text: str) -> list[dict]:
    """Extract valid transaction records from log text, ignoring noise lines."""
    records = []
    for line in text.splitlines():
        match = LOG_PATTERN.match(line.strip())
        if match:
            record = match.groupdict()
            record["price"] = float(record["price"])
            records.append(record)
    return records

#Oppg 3

def remove_price_outliers(prices) -> np.ndarray:
    """Replace values with |x - mean| > 3*std by the median of the valid values."""
    arr = np.asarray(prices, dtype=float).ravel()     # 1D float-array
    mu, sigma = arr.mean(), arr.std()
    is_outlier = np.abs(arr - mu) > 3 * sigma         # boolsk maske
    cleaned = arr.copy()
    if is_outlier.any():
        cleaned[is_outlier] = np.median(arr[~is_outlier])
    return cleaned


def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """Element-wise refund ratio with epsilon to avoid division by zero."""
    sales_count = np.asarray(sales_count, dtype=float)
    refund_count = np.asarray(refund_count, dtype=float)
    return refund_count / (sales_count + 1e-6)

#Oppg 4

def build_dataframe(records: list[dict]) -> pd.DataFrame:
    """List of dicts -> DataFrame with datetime index, sorted chronologically."""
    df = pd.DataFrame(records)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    return df.set_index("timestamp").sort_index()


def hourly_revenue_per_item(df: pd.DataFrame) -> pd.Series:
    """Total price per item in 1-hour windows."""
    return df.groupby("item")["price"].resample("1h").sum()


def status_pivot_table(df: pd.DataFrame) -> pd.DataFrame:
    """Total revenue per item (rows) across status categories (columns)."""
    return pd.pivot_table(df, values="price", index="item", columns="status",
                          aggfunc="sum", fill_value=0)


def item_refund_ratios(df: pd.DataFrame) -> pd.DataFrame:
    """Refund ratio per item = #REFUND / #SUCCESS, via compute_refund_ratio."""
    counts = pd.crosstab(df["item"], df["status"])
    counts = counts.reindex(columns=["SUCCESS", "REFUND", "CANCELLED"], fill_value=0)
    ratios = compute_refund_ratio(counts["SUCCESS"].to_numpy(),
                                  counts["REFUND"].to_numpy())
    return pd.DataFrame({
        "sales_count": counts["SUCCESS"],
        "refund_count": counts["REFUND"],
        "refund_ratio": ratios.round(4),
    })



if __name__ == "__main__":
    log_files = find_log_files("data")
    print(f"{len(log_files)} loggfiler")

    with open("data/store_config.json", encoding="utf-8") as f:
        print("Aktive:", sorted(find_active_registers(json.load(f))))

    records = []
    for path in log_files:
        text, _ = read_log_file(path)
        records.extend(parse_log(text))

    df = build_dataframe(records)
    print(df.head())
    print("Antall 999 før:", (df["price"] == 999).sum())
    df["price"] = remove_price_outliers(df["price"])
    print("Antall 999 etter:", (df["price"] == 999).sum())

    print(status_pivot_table(df))
    print(item_refund_ratios(df))